In [ ]:
import os
import torch
import pandas as pd
import librosa
import evaluate
import matplotlib.pyplot as plt
from datasets import Dataset, Audio

from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer)

from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass
from typing import Any, Dict, List, Union


# B1 Dta Preparation 
print("loading data...")
train_tsv = "../voice_dataset/az/train.tsv"
dev_tsv = "../voice_dataset/az/dev.tsv"
clips_dir = "../voice_dataset/az/clips/"

# read and take a random subset to ensure diverse speaker representation
df_train = pd.read_csv(train_tsv, sep='\t').sample(n=200, random_state=42)
df_dev = pd.read_csv(dev_tsv, sep='\t').sample(n=50, random_state=42)

# Fix column names and build full paths
for df in [df_train, df_dev]:
    df['text_col'] = df['text'] if 'text' in df.columns else df['sentence']
    df['audio'] = df['path'].apply(lambda x: os.path.join(clips_dir, x))

# convert Pandas to Hugging Face Datasets
train_dataset = Dataset.from_pandas(df_train[['audio', 'text_col']])
dev_dataset = Dataset.from_pandas(df_dev[['audio', 'text_col']])

# load processors
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="az", task="transcribe")

def prepare_dataset(batch):
    # openai whisper requires 16kHz 
    audio_array, _ = librosa.load(batch["audio"], sr=16000)

    # turn audio into Mel-Spectrogram 
    batch["input_features"] = processor.feature_extractor(audio_array, sampling_rate=16000).input_features[0]
    # turn text into IDs 
    batch["labels"] = processor.tokenizer(batch["text_col"]).input_ids
    return batch

print("Processing audio...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
dev_dataset = dev_dataset.map(prepare_dataset, remove_columns=dev_dataset.column_names)


# Data Collator

@dataclass
class WhisperCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # add paddings
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = WhisperCollator(processor=processor)


# Metricks (Tracking wer/cer per epoch)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # convert ids to text
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}


# B2 Fine-Tuning (LoRa)

print("loading Model and applying LoRA...")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# freeze the base model parameters
model.requires_grad_(False)

# define LoRA Adapter
config = LoraConfig(
    r=8, # rank - size of the adapter
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # apply only to attention blocks
    lora_dropout=0.05,
    bias="none"
)

# Attach adapter to model
model = get_peft_model(model, config)
model.print_trainable_parameters() 

# Training Settings 
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-az-lora",
    per_device_train_batch_size=2,  
    gradient_accumulation_steps=4,  # accumulate to simulate batch size of 8
    learning_rate=1e-3,             
    num_train_epochs=5,             
    eval_strategy="epoch",    
    save_strategy="epoch",          # save checkpoint every epoch
    load_best_model_at_end=True,    
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,     # generate text during evaluation to calculate WER
    fp16=True,                      # half-precision to save VRAM
    logging_steps=10
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Starting fine-tuning...")
trainer.train()

# Save the final LoRA adapter
model.save_pretrained("./whisper-az-lora-final")
processor.save_pretrained("./whisper-az-lora-final")
print("Training complete! LoRA model saved to ./whisper-az-lora-final")


# B3 Visualisation (graphing loss and wer)
print("Generating training graphs...")
log_history = trainer.state.log_history

epochs = []
train_loss = []
eval_loss = []
eval_wer = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        epochs.append(log["epoch"])
        train_loss.append(log["loss"])
    elif "eval_loss" in log:
        eval_loss.append(log["eval_loss"])
        eval_wer.append(log["eval_wer"])

# Plotting
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(epochs[:len(train_loss)], train_loss, 'b-', label='Train Loss')
ax1.plot(range(1, len(eval_loss) + 1), eval_loss, 'r-', label='Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss', color='k')
ax1.tick_params('y', colors='k')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(range(1, len(eval_wer) + 1), eval_wer, 'g--', label='Validation WER', linewidth=2)
ax2.set_ylabel('WER', color='g')
ax2.tick_params('y', colors='g')
ax2.legend(loc='upper right')

plt.title('Training Progress: Loss and WER over Epochs (LoRA)')
plt.savefig("training_metrics.png")